# שלב 1 — הכנת הנתונים (Porto Taxi Trajectories)

מחברת PySpark לעבודה **מקומית על מדגם**. אותו קוד רץ ללא שינוי על DataProc על הדאטה המלא (רק משנים את הנתיבים ו-`USE_SAMPLE=False`).

**מה המחברת עושה:**
1. טעינת ה-CSV ל-Spark
2. פירוק עמודת `POLYLINE` (JSON של `[[lon,lat],...]`) למערך נקודות
3. ניקוי נתונים פגומים (מתועד)
4. חילוץ פיצ'רים: מספר נקודות, משך נסיעה, מרחק משוער (Haversine)
5. סטטיסטיקות בסיסיות (כולל `approx_percentile`)
6. שמירה כ-**Parquet** (פורמט יעיל לשלבים הבאים)

> כל החישובים הכבדים נעשים עם פונקציות Spark SQL נייטיביות (higher-order functions על מערכים) — בלי Python UDF — כדי שהכול יתבצע מבוזר ויעיל.


## 1. הקמת SparkSession

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import ArrayType, DoubleType

spark = (
    SparkSession.builder
    .appName("porto-taxi-stage1")
    .master("local[*]")                 # מקומי. על Dataproc: להסיר את השורה הזו (YARN ינהל)
    .config("spark.sql.shuffle.partitions", "8")   # מקומי קטן; על הקלאסטר העלו ל-200+
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark


## 2. קונפיגורציה (נתיבים, מדגם/מלא, תיבת גבול פורטו)

In [ ]:
USE_SAMPLE = True   # מקומי = True. ריצה מלאה על Dataproc = False

if USE_SAMPLE:
    INPUT_PATH  = "sample_train.csv"
    OUTPUT_PATH = "out/trips_clean_sample.parquet"
else:
    # על Dataproc — מצביעים ל-GCS
    INPUT_PATH  = "gs://YOUR_BUCKET/raw/train.csv"
    OUTPUT_PATH = "gs://YOUR_BUCKET/clean/trips_clean.parquet"

# תיבת גבול גסה של פורטו (lon/lat) — נקודות מחוץ לה הן ככל הנראה שגיאות GPS
LON_MIN, LON_MAX = -8.75, -8.45
LAT_MIN, LAT_MAX = 41.05, 41.25

# סף מהירות לא-הגיונית בין שתי נקודות עוקבות (15 שניות ביניהן)
MAX_KMH = 200.0
MAX_SEG_KM = MAX_KMH * 15 / 3600   # מרחק מקסימלי סביר בקטע אחד (~0.83 ק"מ)
print(f"max segment km = {MAX_SEG_KM:.3f}")


## 3. טעינת הדאטה הגולמי

In [ ]:
raw = (
    spark.read
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", False)
    .csv(INPUT_PATH)
)
print("raw rows:", raw.count())
raw.printSchema()
raw.select("TRIP_ID", "TAXI_ID", "MISSING_DATA", "POLYLINE").show(2, truncate=60)


## 4. פירוק POLYLINE למערך נקודות

`POLYLINE` הוא מחרוזת JSON. נמיר אותה למערך `array<array<double>>` עם `from_json` (נייטיבי, מבוזר), כאשר לכל נקודה `p[0]=lon, p[1]=lat`.

In [ ]:
point_schema = ArrayType(ArrayType(DoubleType()))

df = raw.withColumn("points", F.from_json("POLYLINE", point_schema))
df = df.withColumn("n_points", F.size("points"))

df.select("TRIP_ID", "n_points").show(3)


## 5. ניקוי נתונים פגומים (סעיף פתוח — מתועד)

ההחלטות והנימוקים:

| בעיה | הסיבה שהיא פוגעת בניתוח | הטיפול |
|------|------------------------|--------|
| `MISSING_DATA = True` | הדאטה עצמו מסומן כחסר | זריקה |
| פוליליין ריק / נקודה אחת | אין מסלול לנתח | `n_points >= 2` |
| נקודה מחוץ לתיבת פורטו | שגיאת GPS שתעוות מרחק/מסלול | פילטר min/max בתוך התיבה |
| "קפיצת" GPS לא-פיזיקלית | קטע יחיד ארוך מנפח את המרחק ויוצר מסלול דמיוני | `max_segment_km <= MAX_SEG_KM` |

נחשב min/max של lon/lat ואת הקטע הארוך ביותר **בלי לפרק את המערך** (higher-order functions).

In [ ]:
# 5.1 פילטרים זולים קודם
df = df.filter((F.col("MISSING_DATA") == "False") & (F.col("n_points") >= 2))

# 5.2 min/max קואורדינטות לכל נסיעה (לבדיקת תיבת גבול)
df = (
    df
    .withColumn("lon_min", F.expr("array_min(transform(points, p -> p[0]))"))
    .withColumn("lon_max", F.expr("array_max(transform(points, p -> p[0]))"))
    .withColumn("lat_min", F.expr("array_min(transform(points, p -> p[1]))"))
    .withColumn("lat_max", F.expr("array_max(transform(points, p -> p[1]))"))
)
df = df.filter(
    (F.col("lon_min") >= LON_MIN) & (F.col("lon_max") <= LON_MAX) &
    (F.col("lat_min") >= LAT_MIN) & (F.col("lat_max") <= LAT_MAX)
)


## 6. חילוץ פיצ'רים: משך, מרחק, קטע מקסימלי (Haversine נייטיבי)

מרחק Haversine בין נקודות עוקבות מחושב עם `zip_with` על המערך המוסט באחד, ו-`aggregate` לסכימה. הכול ב-Spark SQL — מבוזר, בלי UDF.

נוסחה: `d = 2R·asin(sqrt( sin²(Δφ/2) + cosφ₁·cosφ₂·sin²(Δλ/2) ))`, R=6371 ק"מ.

In [ ]:
R_KM = 6371.0

# ביטוי המחזיר מערך של מרחקי הקטעים העוקבים (ק"מ)
seg_km_expr = f"""
zip_with(
    slice(points, 1, size(points)-1),
    slice(points, 2, size(points)-1),
    (a, b) -> 2 * {R_KM} * asin(sqrt(
        pow(sin(radians(b[1]-a[1])/2), 2) +
        cos(radians(a[1])) * cos(radians(b[1])) * pow(sin(radians(b[0]-a[0])/2), 2)
    ))
)
"""

df = (
    df
    .withColumn("seg_km", F.expr(seg_km_expr))
    .withColumn("distance_km", F.expr("aggregate(seg_km, cast(0.0 as double), (acc, x) -> acc + x)"))
    .withColumn("max_segment_km", F.expr("array_max(seg_km)"))
    .withColumn("duration_sec", (F.col("n_points") - 1) * 15)
    .withColumn("avg_kmh", F.when(F.col("duration_sec") > 0,
                                  F.col("distance_km") / (F.col("duration_sec") / 3600.0)))
)

# 6.1 זריקת נסיעות עם קפיצת GPS
df = df.filter(F.col("max_segment_km") <= MAX_SEG_KM)

# DataFrame נקי וסופי של שלב 1
trips = df.select(
    "TRIP_ID", "TAXI_ID", "CALL_TYPE", "ORIGIN_STAND", "TIMESTAMP",
    "points", "n_points", "duration_sec", "distance_km", "avg_kmh"
).cache()

print("clean rows:", trips.count())
trips.select("TRIP_ID","n_points","duration_sec","distance_km","avg_kmh").show(5)


## 7. סטטיסטיקות בסיסיות

In [ ]:
trips.select("n_points", "duration_sec", "distance_km", "avg_kmh").describe().show()

# רבעונים מקורבים (זול ומבוזר) — מתחבר לרעיון ה-Approximate מהפרויקט
trips.select(
    F.expr("approx_percentile(distance_km, array(0.5,0.9,0.99))").alias("distance_km_p50_p90_p99"),
    F.expr("approx_percentile(duration_sec, array(0.5,0.9,0.99))").alias("duration_p50_p90_p99"),
    F.countDistinct("TAXI_ID").alias("n_taxis"),
).show(truncate=False)


## 8. שמירה כ-Parquet (קלט לשלב 2)

In [ ]:
(
    trips
    .write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)
print("saved to:", OUTPUT_PATH)

# בדיקת קריאה חוזרת
check = spark.read.parquet(OUTPUT_PATH)
print("reloaded rows:", check.count())


## הערות להרצה על DataProc (הריצה הסופית)

1. העלו `train.csv` (ועדיף כבר Parquet) ל-GCS bucket.
2. בקובץ קונפיגורציה: `USE_SAMPLE = False` והצביעו ל-`gs://...`.
3. הסירו `.master("local[*]")` — YARN על הקלאסטר ינהל.
4. העלו `spark.sql.shuffle.partitions` ל-200 ומעלה.
5. קלאסטר חסכוני: workers כ-spot/preemptible, `--max-idle`, ולמחוק מיד אחרי הריצה.

**תוצרים של השלב:** DataFrame מתועד (`OUTPUT_PATH`) + סטטיסטיקות בסיסיות.
